# 🏥 BilingualExtractor - Interactive Medical Term Extraction
## Arabic-English Parallel Corpus Builder

---

### 📋 Table of Contents
1. [Setup & Installation](#setup)
2. [Load Sample Medical Text](#load)
3. [Term Extraction](#extraction)
4. [Analysis & Statistics](#analysis)
5. [Export Results](#export)
6. [Advanced: Custom Patterns](#advanced)

---

## 1️⃣ Setup & Installation <a name="setup"></a>

In [ ]:
# Install required packages
!pip install sentence-transformers scipy pandas openpyxl -q

print('✅ All dependencies installed successfully!')

In [ ]:
import re
import json
import csv
import logging
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Tuple, Optional, Set
from collections import Counter, defaultdict
from difflib import SequenceMatcher
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

# Configure plotting
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/chinese/NotoSansSC[wght].ttf')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Noto Sans SC']
plt.rcParams['axes.unicode_minus'] = False

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('BilingualExtractor')

print('✅ All modules imported successfully!')

## 🧩 Core Classes

In [ ]:
@dataclass
class TermPair:
    """Bilingual term pair (Arabic - English)."""
    arabic_term: str
    english_term: str
    frequency: int = 1
    confidence: float = 0.0
    source_page: int = -1
    category: str = "general"
    context_ar: str = ""
    context_en: str = ""
    variants_ar: List[str] = field(default_factory=list)
    variants_en: List[str] = field(default_factory=list)

    def to_dict(self) -> Dict:
        return asdict(self)


@dataclass
class ExtractionConfig:
    """Configuration for term extraction."""
    min_term_length_ar: int = 2
    min_term_length_en: int = 2
    min_frequency: int = 1
    min_confidence: float = 0.3
    deduplication_threshold: float = 0.85
    include_context: bool = True

    def to_dict(self) -> Dict:
        return asdict(self)


print('✅ Data classes defined!')

In [ ]:
class TextPreprocessor:
    """Preprocesses Arabic and English text."""

    def __init__(self):
        self._arabic_chars = set()
        for code in range(0x0600, 0x06FF + 1):
            self._arabic_chars.add(chr(code))
        for code in range(0x0750, 0x077F + 1):
            self._arabic_chars.add(chr(code))

    def normalize_arabic(self, text: str) -> str:
        if not text:
            return ""
        text = text.replace("\u0640", "")  # Remove tatweel
        text = text.replace("ة", "ه")       # Normalize taa marbuta
        text = text.replace("ﻯ", "ى")       # Normalize alef maqsura
        # Remove diacritics
        diacritics = re.compile(r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E4\u06E7\u06E8\u06EA-\u06ED]')
        text = diacritics.sub('', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def normalize_english(self, text: str) -> str:
        if not text:
            return ""
        text = text.lower().strip()
        text = re.sub(r'[^a-z0-9\s\-]', '', text)
        return text

    def clean_ocr_errors(self, text: str) -> str:
        if not text:
            return ""
        ocr_fixes = {"ﻻ": "لا", "ﻼ": "لا", "ـ": ""}
        for wrong, correct in ocr_fixes.items():
            text = text.replace(wrong, correct)
        return text.strip()

    def detect_language(self, text: str) -> str:
        if not text:
            return "unknown"
        ar_count = sum(1 for c in text if c in self._arabic_chars)
        en_count = sum(1 for c in text if c.isascii() and c.isalpha())
        total = ar_count + en_count
        if total == 0:
            return "unknown"
        if ar_count / total > 0.7:
            return "arabic"
        elif en_count / total > 0.7:
            return "english"
        return "mixed"


preprocessor = TextPreprocessor()
print('✅ TextPreprocessor ready!')

# Test
test_ar = "الكسر المركب في عظم الفخذ"
print(f"   Normalize Arabic: '{test_ar}' → '{preprocessor.normalize_arabic(test_ar)}'")
print(f"   Detect language: '{test_ar}' → {preprocessor.detect_language(test_ar)}")

In [ ]:
# Medical terminology resources
DIALECT_MAPPINGS = {
    "الكُسر": ["الكَسر", "الانكسار", "الكسر"],
    "الشلل": ["الفالج", "الرعاش", "الخدر"],
    "التهاب": ["ورم", "احتقان"],
}

TERMINOLOGY_EVOLUTION = {
    "الفالج": "الشلل (Paralysis)",
    "الاستسقاء": "الوذمة (Edema)",
    "اليرقان": "اليرقان/الصفراء (Jaundice)",
}

MEDICAL_CATEGORIES = {
    "anatomy": ["عظم", "مفصل", "عضلة", "وتر", "غضروف", "أربطة", "نسيج", "عمود فقري"],
    "orthopedics": ["كسر", "خلع", "تهتك", "انزلاق غضروفي", "التهاب مفاصل", "روماتيزم", "هشاشة عظام"],
    "cardiology": ["قلب", "شريان", "وريد", "صمام", "جلطة", "احتشاء", "قصور"],
    "neurology": ["عصب", "دماغ", "حبل شوكي", "شلل", "تشنج", "صرع", "صداع"],
    "surgery": ["عملية جراحية", "شق", "خياطة", "تنظير", "استئصال", "زراعة", "مخدر"],
    "internal_medicine": ["التهاب", "عدوى", "بكتيريا", "فيروس", "مناعة", "حساسية", "سرطان", "سكري"],
}

print(f'✅ Medical resources loaded!')
print(f'   {len(DIALECT_MAPPINGS)} dialect mappings')
print(f'   {len(TERMINOLOGY_EVOLUTION)} historical evolutions')
print(f'   {len(MEDICAL_CATEGORIES)} medical categories')

In [ ]:
class PatternExtractor:
    """Extracts medical terms using regex patterns."""

    def __init__(self, config=None):
        self.config = config or ExtractionConfig()
        self.preprocessor = TextPreprocessor()
        self._compile_patterns()

    def _compile_patterns(self):
        # Arabic patterns
        self.arabic_patterns = [
            re.compile(r'(?:ال|لل)\s*[\u0600-\u06FF]{2,15}(?:\s+[\u0600-\u06FF]{2,10})*'),
            re.compile(r'التهاب\s+[\u0600-\u06FF]{2,15}(?:\s+[\u0600-\u06FF]{2,10})*'),
            re.compile(r'[\u0600-\u06FF]{2,15}(?:\s+و\s+[\u0600-\u06FF]{2,15})+'),
        ]

        # English patterns
        self.english_patterns = [
            re.compile(r'[a-zA-Z]{2,}(?:\s+[a-zA-Z]{2,}){0,3}'),
            re.compile(r'[a-zA-Z]{2,}(?:-[a-zA-Z]+){1,3}'),
            re.compile(r'\(([a-zA-Z][a-zA-Z\s\-]{2,40})\)'),
        ]

        # Bilingual pair patterns
        self.bilingual_pair_patterns = [
            re.compile(r'([\u0600-\u06FF][\u0600-\u06FF\s]{2,30})\s*[-–—:：\(\[]\s*([A-Za-z][A-Za-z\s\-]{2,30})'),
            re.compile(r'([A-Za-z][A-Za-z\s\-]{2,30})\s*[-–—:：\[\(]\s*([\u0600-\u06FF][\u0600-\u06FF\s]{2,30})'),
        ]

    def extract_bilingual_pairs(self, text: str) -> List[Tuple[str, str]]:
        text = self.preprocessor.clean_ocr_errors(text)
        pairs, seen = [], set()
        for pattern in self.bilingual_pair_patterns:
            for match in pattern.finditer(text):
                ar = match.group(1).strip()
                en = match.group(2).strip()
                ar_n = self.preprocessor.normalize_arabic(ar)
                en_n = self.preprocessor.normalize_english(en)
                if ar_n and en_n and (ar_n, en_n) not in seen:
                    seen.add((ar_n, en_n))
                    pairs.append((ar_n, en_n))
        return pairs

    def extract_arabic_terms(self, text: str) -> List[str]:
        text = self.preprocessor.clean_ocr_errors(text)
        terms = set()
        for p in self.arabic_patterns:
            for m in p.finditer(text):
                t = m.group().strip()
                n = self.preprocessor.normalize_arabic(t)
                if n:
                    terms.add(n)
        return list(terms)

    def extract_english_terms(self, text: str) -> List[str]:
        terms = set()
        for p in self.english_patterns:
            for m in p.finditer(text):
                t = m.group().strip()
                n = self.preprocessor.normalize_english(t)
                if n:
                    terms.add(n)
        return list(terms)


pattern_extractor = PatternExtractor()
print('✅ PatternExtractor ready!')

In [ ]:
class TermExtractor:
    """Main bilingual medical term extractor."""

    def __init__(self, config=None):
        self.config = config or ExtractionConfig()
        self.preprocessor = TextPreprocessor()
        self.pattern_extractor = PatternExtractor(config)
        self._raw_text = ""
        self._term_pairs: Dict[Tuple[str, str], TermPair] = {}
        self._arabic_freq = Counter()
        self._english_freq = Counter()

    def load_text(self, text: str) -> 'TermExtractor':
        self._raw_text = text
        return self

    def load_text_file(self, path: str) -> 'TermExtractor':
        with open(path, 'r', encoding='utf-8') as f:
            self._raw_text = f.read()
        return self

    def _categorize(self, term: str) -> str:
        for cat, terms in MEDICAL_CATEGORIES.items():
            for t in terms:
                if t in term or term in t:
                    return cat
        return "general"

    def _calc_confidence(self, ar: str, en: str, co: int) -> float:
        score = 0.0
        score += min(co / 5.0, 1.0) * 0.3  # frequency
        ar_l, en_l = len(ar), len(en)
        if ar_l > 0 and en_l > 0:
            score += (min(ar_l, en_l) / max(ar_l, en_l)) * 0.2  # balance
        medical_kw = {"itis", "osis", "oma", "ectomy", "otomy", "التهاب", "كسر", "مفصل", "عظم"}
        if any(k in ar.lower() or k in en.lower() for k in medical_kw):
            score += 0.2  # medical bonus
        score += 0.3  # pattern match bonus
        return round(min(score, 1.0), 4)

    def extract_terms(self) -> List[TermPair]:
        if not self._raw_text:
            return []

        import time
        start = time.time()

        pairs = self.pattern_extractor.extract_bilingual_pairs(self._raw_text)
        ar_terms = self.pattern_extractor.extract_arabic_terms(self._raw_text)
        en_terms = self.pattern_extractor.extract_english_terms(self._raw_text)

        self._arabic_freq.update(ar_terms)
        self._english_freq.update(en_terms)

        for ar, en in pairs:
            key = (ar, en)
            if key in self._term_pairs:
                self._term_pairs[key].frequency += 1
            else:
                self._term_pairs[key] = TermPair(arabic_term=ar, english_term=en)

        for pair in self._term_pairs.values():
            pair.category = self._categorize(pair.arabic_term)
            pair.confidence = self._calc_confidence(
                pair.arabic_term, pair.english_term, pair.frequency)

        results = sorted(self._term_pairs.values(), key=lambda t: t.confidence, reverse=True)
        results = [t for t in results if t.confidence >= self.config.min_confidence]

        elapsed = time.time() - start
        print(f'Extraction completed in {elapsed:.2f}s - Found {len(results)} pairs')
        return results

    def export_to_csv(self, path: str) -> str:
        terms = sorted(self._term_pairs.values(), key=lambda t: t.confidence, reverse=True)
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'w', encoding='utf-8-sig', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Arabic Term', 'English Term', 'Frequency', 'Confidence', 'Category'])
            for t in terms:
                writer.writerow([t.arabic_term, t.english_term, t.frequency, t.confidence, t.category])
        return path

    def export_to_json(self, path: str) -> str:
        terms = sorted(self._term_pairs.values(), key=lambda t: t.confidence, reverse=True)
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        data = {"terms": [t.to_dict() for t in terms]}
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        return path


print('✅ TermExtractor class ready!')

## 2️⃣ Load Sample Medical Text <a name="load"></a>

In [ ]:
# Sample bilingual medical text
sample_medical_text = """
الكتاب المرجعي في جراحة العظام والكسور
Orthopedic Surgery Reference Book

الفصل الاول: مقدمة في جراحة العظام
Chapter 1: Introduction to Orthopedic Surgery

الكسر (Fracture) هو انقطاع في استمرارية العظم الناتج عن اصابة او مرض.
A fracture is a discontinuity in bone continuity resulting from injury or disease.

انواع الكسور (Types of Fractures):
1. الكسر البسيط (Simple Fracture) - كسر لا يخترق الجلد
2. الكسر المركب (Compound Fracture) - كسر يخترق الجلد
3. كسر ملتوي (Spiral Fracture) - ناتج عن قوة الالتواء
4. كسر مضغوط (Compression Fracture) - ناتج عن ضغط مباشر
5. الكسر المفصلي (Articular Fracture) - يمتد الى سطح المفصل

التهاب المفاصل (Arthritis) هو التهاب يصيب المفاصل causing pain and stiffness.
انواع التهاب المفاصل تشمل:
- التهاب المفاصل الروماتويدي (Rheumatoid Arthritis)
- التهاب المفاصل التنكسي (Osteoarthritis)
- التهاب المفاصل النقرسي (Gouty Arthritis)

الانزلاق الغضروفي (Herniated Disc) يحدث عندما ينتفخ النواة اللبية
للقرص الفقري (Spinal Disc) ويضغط على الاعصاب.

هشاشة العظام (Osteoporosis) هي حالة ضعف العظام التي تزيد من خطر الكسر.
Bone density test is used to diagnose osteoporosis.

عملية استئصال الغدة الدرقية (Thyroidectomy) involve removal of thyroid gland.
The surgeon performs thyroidectomy under general anesthesia.

شلل الاطفال (Poliomyelitis) كان يسمى قديما بالفالج.
Paralysis can affect one side (hemiplegia) or both sides (paraplegia).

الفحص بالاشعة السينية (X-ray Examination) هو اول خطوة في تشخيص الكسور.
التصوير بالرنين المغناطيسي (MRI) gives detailed images of soft tissues.

علاج كسور العظام (Bone Fracture Treatment) يتضمن عدة طرق:
- الجبيرة (Cast) - تثبيت الكسر
- التثبيت الداخلي (Internal Fixation) - باستخدام مسامير واضواح
- التثبيت الخارجي (External Fixation) - using pins and frames

العمود الفقري (Spine) يتكون من 33 فقرة.
فقرات العمود الفقري (Vertebrae) تحمي الحبل الشوكي (Spinal Cord).

اصابات الكتف (Shoulder Injuries) تشمل:
- خلع الكتف (Shoulder Dislocation)
- تمزق الكفة المدورة (Rotator Cuff Tear)
- التهاب الكفة المدورة (Rotator Cuff Tendinitis)

جراحة الركبة (Knee Surgery) تشمل:
- استبدال الركبة (Knee Replacement)
- تنظير الركبة (Knee Arthroscopy)
- اصلاح الرباط الصليبي (ACL Reconstruction)

المفاصل (Joints) are where two or more bones meet.
الاغشية الزلالية (Synovial Membranes) line the joints.
سائل المفصل (Synovial Fluid) lubricates the joint.
"""

print(f'Loaded sample text: {len(sample_medical_text)} characters')
print(f'Lines: {len(sample_medical_text.strip().splitlines())}')

## 3️⃣ Term Extraction <a name="extraction"></a>

In [ ]:
# Create extractor and run
extractor = TermExtractor()
extractor.load_text(sample_medical_text)
terms = extractor.extract_terms()

print(f'\nTotal term pairs extracted: {len(terms)}')

In [ ]:
# Display results in a nice table
df = pd.DataFrame([{
    'Arabic Term': t.arabic_term,
    'English Term': t.english_term,
    'Frequency': t.frequency,
    'Confidence': f'{t.confidence:.2%}',
    'Category': t.category
} for t in terms])

display(HTML('<h3>Extracted Medical Terms</h3>'))
display(df.style.set_properties(**{
    'text-align': 'right',
    'direction': 'rtl'
}).background_gradient(subset=['Confidence'], cmap='RdYlGn'))

## 4️⃣ Analysis & Statistics <a name="analysis"></a>

In [ ]:
# Statistics
categories = Counter(t.category for t in terms)
confidence_levels = {
    'High (>= 0.7)': sum(1 for t in terms if t.confidence >= 0.7),
    'Medium (0.4-0.7)': sum(1 for t in terms if 0.4 <= t.confidence < 0.7),
    'Low (< 0.4)': sum(1 for t in terms if t.confidence < 0.4),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Category distribution
ax1 = axes[0]
bars = ax1.bar(categories.keys(), categories.values(), color='steelblue', edgecolor='navy')
ax1.set_title('Terms by Medical Category', fontsize=14, fontweight='bold')
ax1.set_ylabel('Number of Terms')
ax1.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, categories.values()):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, str(val),
             ha='center', va='bottom', fontweight='bold')

# Confidence distribution
ax2 = axes[1]
colors = ['#2ecc71', '#f39c12', '#e74c3c']
wedges, texts, autotexts = ax2.pie(
    confidence_levels.values(),
    labels=confidence_levels.keys(),
    colors=colors, autopct='%1.1f%%',
    startangle=90
)
ax2.set_title('Confidence Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\n📊 Summary Statistics:')
print(f'   Total terms: {len(terms)}')
print(f'   Categories: {dict(categories)}')
print(f'   Confidence: {dict(confidence_levels)}')

In [ ]:
# Term frequency analysis
from wordcloud import WordCloud

# Try to create word clouds
try:
    ar_text = ' '.join(t.arabic_term for t in terms)
    en_text = ' '.join(t.english_term for t in terms)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Arabic word cloud
    wc_ar = WordCloud(width=800, height=400, background='white',
                      font_path='/usr/share/fonts/truetype/chinese/NotoSansSC[wght].ttf')
    wc_ar.generate(ar_text)
    axes[0].imshow(wc_ar, interpolation='bilinear')
    axes[0].set_title('Arabic Medical Terms', fontsize=14, fontweight='bold')
    axes[0].axis('off')

    # English word cloud
    wc_en = WordCloud(width=800, height=400, background='white')
    wc_en.generate(en_text)
    axes[1].imshow(wc_en, interpolation='bilinear')
    axes[1].set_title('English Medical Terms', fontsize=14, fontweight='bold')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()
except ImportError:
    print('WordCloud not available. Install with: pip install wordcloud')
except Exception as e:
    print(f'WordCloud generation failed: {e}')

## 5️⃣ Export Results <a name="export"></a>

In [ ]:
# Export to CSV
csv_path = extractor.export_to_csv('/tmp/medical_terms.csv')
json_path = extractor.export_to_json('/tmp/medical_terms.json')

print('📁 Exported Files:')
print(f'   CSV: {csv_path}')
print(f'   JSON: {json_path}')

# Download from Colab
try:
    from google.colab import files
    files.download(csv_path)
    files.download(json_path)
    print('\n✅ Files downloaded to your local machine!')
except ImportError:
    print('\nℹ️ Running locally - files saved to /tmp/')

In [ ]:
# Export to Excel with formatting
excel_data = pd.DataFrame([{
    'المصطلح العربي': t.arabic_term,
    'المصطلح الإنجليزي': t.english_term,
    'التكرار': t.frequency,
    'نسبة الثقة': t.confidence,
    'التصنيف': t.category,
} for t in sorted(terms, key=lambda x: x.confidence, reverse=True)])

excel_path = '/tmp/medical_terms.xlsx'

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    excel_data.to_excel(writer, sheet_name='Extracted Terms', index=False)

    # Stats sheet
    stats_df = pd.DataFrame({
        'Metric': ['Total Terms', 'High Confidence', 'Medium Confidence',
                   'Low Confidence', 'Categories'],
        'Value': [len(terms), confidence_levels['High (>= 0.7)'],
                  confidence_levels['Medium (0.4-0.7)'],
                  confidence_levels['Low (< 0.4)'],
                  len(categories)]
    })
    stats_df.to_excel(writer, sheet_name='Statistics', index=False)

print(f'✅ Excel file saved: {excel_path}')

try:
    from google.colab import files
    files.download(excel_path)
except ImportError:
    pass

## 6️⃣ Advanced: Custom Patterns & Upload <a name="advanced"></a>

In [ ]:
# Upload your own textbook file (Colab only)
try:
    from google.colab import files

    print('📤 Upload your bilingual medical textbook (TXT format):')
    uploaded = files.upload()

    for filename, content in uploaded.items():
        text = content.decode('utf-8')
        print(f'\n📖 Processing: {filename} ({len(text)} chars)')

        custom_extractor = TermExtractor()
        custom_extractor.load_text(text)
        custom_terms = custom_extractor.extract_terms()

        custom_df = pd.DataFrame([{
            'المصطلح العربي': t.arabic_term,
            'المصطلح الإنجليزي': t.english_term,
            'نسبة الثقة': f'{t.confidence:.2%}',
            'التصنيف': t.category
        } for t in custom_terms])

        display(custom_df.head(20))

        # Export
        out_csv = f'/tmp/{filename}_terms.csv'
        custom_extractor.export_to_csv(out_csv)
        files.download(out_csv)

except ImportError:
    print('ℹ️ File upload only available in Google Colab.')
    print('   To use locally, pass the file path to load_text_file():')
    print('   extractor.load_text_file("path/to/textbook.txt")')

In [ ]:
# Advanced: Add custom medical domain patterns
custom_domain_terms = {
    "cardiology": ["قلب", "شريان", "وريد", "صمام", "جلطة", "احتشاء",
                    "سدة", "قصور", "رعاش", "خفقان", "Arrhythmia",
                    "Myocardial Infarction", "Heart Failure", "Valve"],
    "neurology": ["عصب", "دماغ", "حبل شوكي", "شلل", "تشنج", "صرع",
                   "صداع", "شقيقة", "خدر", "Neuralgia", "Neuropathy",
                   "Epilepsy", "Migraine"],
}

print('📚 Custom Medical Domain Patterns:')
for domain, terms_list in custom_domain_terms.items():
    print(f'\n   {domain.upper()}:')
    for t in terms_list[:5]:
        print(f'     - {t}')
    print(f'     ... and {len(terms_list) - 5} more')

print('\n✅ You can extend these patterns for your specific medical domain!')

In [ ]:
print('=' * 60)
print('  BilingualExtractor Notebook - Complete!')
print('=' * 60)
print()
print('What you can do next:')
print('  1. Upload your own medical textbooks')
print('  2. Adjust confidence thresholds')
print('  3. Add custom medical domain patterns')
print('  4. Export to different formats (CSV, JSON, Excel, TMX)')
print('  5. Integrate with Sentence Alignment (next sprint)')
print()
print('Questions? Open an issue on the repository.')